# Part 3. Core로 데이터 다루기

지금까지는 `text()`로 raw SQL을 실행하거나, 메타데이터로 스키마를 정의했다. 이제 SQLAlchemy의 진짜 강점인 **SQL Expression Language**를 다룬다. Python 함수와 객체로 SQL을 조립하는 방식이다. 5장의 `text()`와 달리, 이 방식은 타입 안전하고, 재사용 가능하며, 데이터베이스 방언을 자동으로 처리한다.

---

## 8장. INSERT 구문

#### users 테이블 생성

In [1]:
from sqlalchemy import (
    create_engine, MetaData, Table, Column,
    Integer, String, DateTime, ForeignKey, func, 
)

engine = create_engine("sqlite:///:memory:", echo=True)
metadata = MetaData()

users = Table(
    "users",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String(50), nullable=False),
    Column("email", String(120), unique=True),
    Column("created_at", DateTime, server_default=func.now()),
)

metadata.create_all(engine)

2026-06-01 10:13:31,499 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 10:13:31,500 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-01 10:13:31,500 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 10:13:31,501 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("users")
2026-06-01 10:13:31,501 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 10:13:31,503 INFO sqlalchemy.engine.Engine 
CREATE TABLE users (
	id INTEGER NOT NULL, 
	name VARCHAR(50) NOT NULL, 
	email VARCHAR(120), 
	created_at DATETIME DEFAULT CURRENT_TIMESTAMP, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)


2026-06-01 10:13:31,503 INFO sqlalchemy.engine.Engine [no key 0.00038s] ()
2026-06-01 10:13:31,504 INFO sqlalchemy.engine.Engine COMMIT


### 8.1 `insert()` 기본 사용법

`insert()` 함수는 `Insert` 객체를 만들어 INSERT 문을 표현한다.

In [3]:
from sqlalchemy import insert

# INSERT 구문 생성
insert_users = insert(users)
print(type(insert_users))
print('🟦', insert_users)



<class 'sqlalchemy.sql.dml.Insert'>
🟦 INSERT INTO users (id, name, email, created_at) VALUES (:id, :name, :email, :created_at)


In [4]:
stmt = insert(users).values(name="Alice", email="alice@example.com")
print(type(stmt))
print('🟦', stmt)

<class 'sqlalchemy.sql.dml.Insert'>
🟦 INSERT INTO users (name, email) VALUES (:name, :email)


In [ ]:
# 주목할 점은 `stmt` 자체는 단지 SQL을 표현한 객체일 뿐, 
# 아직 실행되지 않았다는 것이다. 출력해보면 어떤 SQL이 생성될지 확인할 수 있다. 

In [5]:
# ↓실행은 `connection.execute()`로 한다.

with engine.begin() as conn:
    result = conn.execute(stmt)
    print(result.inserted_primary_key)

# `result.inserted_primary_key`는 새로 삽입된 행의 기본 키 값을 튜플로 반환한다. 자동 증가 컬럼의 값을 알아낼 때 유용하다.

2026-06-01 10:19:48,465 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 10:19:48,466 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email) VALUES (?, ?)
2026-06-01 10:19:48,467 INFO sqlalchemy.engine.Engine [generated in 0.00050s] ('Alice', 'alice@example.com')
(1,)
2026-06-01 10:19:48,469 INFO sqlalchemy.engine.Engine COMMIT


### 8.2 `values()` 없이 파라미터로 전달하기

`values()`를 호출하지 않고, `execute()`의 두 번째 인자로 데이터를 전달하는 방식도 자주 사용된다.

In [6]:
with engine.begin() as conn:
    result = conn.execute(
        insert(users),
        {"name": "Bob", "email": "bob@example.com"},
    )
    print(result.inserted_primary_key)


2026-06-01 10:22:02,453 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 10:22:02,454 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email) VALUES (?, ?)
2026-06-01 10:22:02,455 INFO sqlalchemy.engine.Engine [generated in 0.00061s] ('Bob', 'bob@example.com')
(2,)
2026-06-01 10:22:02,455 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
# 이 방식은 다중 행 INSERT에 특히 유용하다. 
# 단일 행이라도 일관성을 위해 이 스타일을 선호하는 사람도 많다. 
# 어느 쪽을 쓰든 SQLAlchemy는 동일한 SQL을 생성한다.


### 8.3 다중 행 INSERT

여러 행을 한 번에 삽입하려면 딕셔너리 리스트를 전달한다.

In [7]:
with engine.begin() as conn:
    result = conn.execute(
        insert(users),
        [
            {"name": "Charlie", "email": "charlie@example.com"},
            {"name": "Diana", "email": "diana@example.com"},
            {"name": "Eve", "email": "eve@example.com"},            
        ],
    )
    print(result.rowcount)
    

2026-06-01 10:23:48,852 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 10:23:48,854 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email) VALUES (?, ?)
2026-06-01 10:23:48,854 INFO sqlalchemy.engine.Engine [generated in 0.00086s] [('Charlie', 'charlie@example.com'), ('Diana', 'diana@example.com'), ('Eve', 'eve@example.com')]
3
2026-06-01 10:23:48,855 INFO sqlalchemy.engine.Engine COMMIT


SQLAlchemy 2.0의 **`insertmanyvalues`** 기능 덕분에, RETURNING을 지원하는 백엔드(PostgreSQL, SQLite, MariaDB, Oracle 등)에서는 내부적으로 `INSERT ... VALUES (...), (...), (...)` 형태로 묶어 단일 SQL로 실행한다. 1.x 대비 INSERT 성능이 극적으로 개선된 핵심 기능이다.

### 8.4 RETURNING 절

INSERT 후 삽입된 행의 정보를 받아오고 싶을 때 `returning()`을 사용한다. PostgreSQL, SQLite, MariaDB, Oracle, SQL Server에서 지원되며, MySQL은 지원하지 않는다.

In [10]:
stmt = (
    insert(users)
    .values(name="Frank", email="frank@example.com")
    .returning(users.c.id, users.c.created_at)
)

with engine.begin() as conn:
    result = conn.execute(stmt)
    row = result.one()  # insert 된 행 한개
    print(row.id, row.created_at)
    

2026-06-01 10:27:26,411 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 10:27:26,412 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email) VALUES (?, ?) RETURNING id, created_at
2026-06-01 10:27:26,412 INFO sqlalchemy.engine.Engine [generated in 0.00046s] ('Frank', 'frank@example.com')
6 2026-06-01 01:27:26
2026-06-01 10:27:26,415 INFO sqlalchemy.engine.Engine COMMIT


여러 컬럼을 받으려면 `returning()`에 여러 컬럼을 나열한다. 전체 행을 받고 싶다면 `returning(users)`처럼 테이블을 통째로 넘긴다.

RETURNING은 다중 행 INSERT에서도 동작한다.

In [ ]:
stmt = insert(users) 🔹TODO

with engine.begin() as conn:
    result = conn.execute(
        stmt,
        [
            {"name": "Grace", "email": "grace@example.com"},
            {"name": "Henry", "email": "henry@example.com"},
        ],
    )
    for row in result:
        print('🟩', row.id, row.name)

### 8.5 INSERT ... FROM SELECT

다른 테이블의 데이터를 SELECT해서 그대로 INSERT하는 패턴이다.

```python
from sqlalchemy import select

archived_users = Table(
    "archived_users",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String(50)),
    Column("email", String(120)),
)

# users에서 데이터를 골라 archived_users로 복사
select_stmt = select(users.c.id, users.c.name, users.c.email).where(
    users.c.created_at < some_date
)
insert_stmt = insert(archived_users).from_select(
    ["id", "name", "email"], select_stmt
)
```

이는 데이터 복사, 백업, 비정규화 작업 등에서 유용하다.

### 8.6 컬럼 접근: `table.c.column_name`

위 예제에서 `users.c.id` 같은 표기가 나왔다. `.c`는 `columns`의 줄임말로, 테이블의 컬럼 컬렉션이다.

```python
users.c.id        # id 컬럼
users.c.name      # name 컬럼
users.columns.id  # 동일 (긴 이름)
```

이 컬럼 객체들은 단순한 이름이 아니라 표현식 객체다. 비교 연산자나 함수를 적용하면 SQL 조건이 만들어진다. 다음 장에서 본격적으로 활용한다.

---

## 9장. SELECT 구문 기초

### 9.1 `select()` 기본

`select()` 함수가 SELECT 문의 시작점이다. 인자로 가져올 컬럼이나 테이블을 넘긴다.

In [11]:
from sqlalchemy import select

# 전체 테이블
stmt = select(users)
print(type(stmt))
print(stmt)

<class 'sqlalchemy.sql.selectable.Select'>
SELECT users.id, users.name, users.email, users.created_at 
FROM users


In [12]:
# 특정 컬럼
stmt = select(users.c.name, users.c.email)
print(stmt)

SELECT users.name, users.email 
FROM users


In [13]:
# 실행은 `connection.execute()`로 한다.
with engine.connect() as conn:
    result = conn.execute(select(users))
    for row in result:
        print('🟩', row.id, row.name, row.email)

2026-06-01 10:32:36,776 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 10:32:36,777 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FROM users
2026-06-01 10:32:36,777 INFO sqlalchemy.engine.Engine [generated in 0.00108s] ()
🟩 1 Alice alice@example.com
🟩 2 Bob bob@example.com
🟩 3 Charlie charlie@example.com
🟩 4 Diana diana@example.com
🟩 5 Eve eve@example.com
🟩 6 Frank frank@example.com
2026-06-01 10:32:36,778 INFO sqlalchemy.engine.Engine ROLLBACK


### 9.2 `WHERE` 절: `.where()`

조건을 추가하려면 `.where()`를 체이닝한다.

In [14]:
stmt = select(users).where(users.c.name == "Alice")
print(stmt)


SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.name = :name_1


여기서 `users.c.name == "Alice"`는 **Python의 비교 연산자가 오버로딩된 SQL 표현식**이다. 단순한 비교가 아니라 SQL 조건 객체를 만들어낸다. `print(users.c.name == "Alice")`를 해보면 `users.name = :name_1`이 출력된다.

여러 조건을 AND로 결합하려면 `.where()`를 여러 번 호출하거나, 쉼표로 구분한다.

In [15]:
# 다음 두 방식은 동일하다
stmt = select(users).where(users.c.name == "Alice", users.c.id > 0)
print(stmt)

stmt = select(users).where(users.c.name == "Alice").where(users.c.id > 0)
print(stmt)


SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.name = :name_1 AND users.id > :id_1
SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.name = :name_1 AND users.id > :id_1


In [18]:
# 명시적으로 `and_()`, `or_()`를 사용할 수도 있다.

from sqlalchemy import and_, or_

stmt = select(users).where(
    or_(
        users.c.name == "Alice",
        and_(users.c.name == "Bob", users.c.id > 10),
    )
)
print(stmt)


SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.name = :name_1 OR users.name = :name_2 AND users.id > :id_1


### 9.3 비교 연산자와 SQL 함수

Python 연산자와 SQL의 대응은 다음과 같다.

| Python 표현 | SQL |
|---|---|
| `users.c.id == 5` | `id = 5` |
| `users.c.id != 5` | `id != 5` |
| `users.c.id > 5` | `id > 5` |
| `users.c.id.in_([1, 2, 3])` | `id IN (1, 2, 3)` |
| `users.c.name.like("A%")` | `name LIKE 'A%'` |
| `users.c.name.ilike("a%")` | `name ILIKE 'a%'` (대소문자 무시) |
| `users.c.email.is_(None)` | `email IS NULL` |
| `users.c.email.is_not(None)` | `email IS NOT NULL` |
| `users.c.id.between(1, 10)` | `id BETWEEN 1 AND 10` |
| `~(users.c.id == 5)` | `NOT (id = 5)` |

**중요**: NULL 비교에는 반드시 `is_()` / `is_not()`을 써야 한다. `users.c.email == None`은 동작하긴 하지만(SQLAlchemy가 알아서 `IS NULL`로 변환) 의도가 명확한 `.is_()` 사용이 권장된다.

### 9.4 정렬: `.order_by()`

`order_by()`로 정렬한다.

In [19]:
stmt = select(users).order_by(users.c.created_at)   # ASC (기본)
print(stmt)

SELECT users.id, users.name, users.email, users.created_at 
FROM users ORDER BY users.created_at


In [20]:
stmt = select(users).order_by(users.c.created_at.desc())  # DESC
print(stmt)

SELECT users.id, users.name, users.email, users.created_at 
FROM users ORDER BY users.created_at DESC


In [21]:
stmt = select(users).order_by(users.c.name.asc(), users.c.id.desc()) # 여러 컬럼 정렬
print(stmt)

SELECT users.id, users.name, users.email, users.created_at 
FROM users ORDER BY users.name ASC, users.id DESC


### 9.5 결과 개수 제한: `.limit()`, `.offset()`

페이징을 위한 `limit()`과 `offset()`이다.

In [22]:
# 처음 10개
stmt = select(users).order_by(users.c.id).limit(10)
print('🟩', stmt)

🟩 SELECT users.id, users.name, users.email, users.created_at 
FROM users ORDER BY users.id
 LIMIT :param_1


In [23]:
# 10개를 건너뛰고 다음 10개 (2페이지)
stmt = select(users).order_by(users.c.id).limit(10).offset(10)
print('🟩', stmt)

🟩 SELECT users.id, users.name, users.email, users.created_at 
FROM users ORDER BY users.id
 LIMIT :param_1 OFFSET :param_2


In [24]:
# 대규모 데이터에서 큰 OFFSET은 성능에 좋지 않다. 이런 경우 키셋 페이지네이션(cursor-based)을 고려한다.

# id > last_seen_id 방식의 페이징
last_seen_id = 4
stmt = select(users).where(users.c.id > last_seen_id).order_by(users.c.id).limit(10)  # <-🔹TODO
print('🟩', stmt)

🟩 SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.id > :id_1 ORDER BY users.id
 LIMIT :param_1


### 9.6 결과 가져오기: 다양한 방법

4장에서 본 `Result` 객체의 메서드를 다시 정리한다.

In [25]:
with engine.connect() as conn:
    # 단일 행
    row = conn.execute(select(users).where(users.c.id == 1)).one()
    print('🟩', row)
    
    row = conn.execute(select(users).where(users.c.id == 1)).one_or_none()
    print('🟩', row)
    
    # 단일 스칼라 값
    name = conn.execute(select(users.c.name).where(users.c.id == 1)).scalar_one()
    print('🟩', name)
    
    # 모든 행
    rows = conn.execute(select(users)).all()
    print('🟩', rows)
    
    # 스칼라 컬럼들의 리스트
    # `.scalars()`는 결과의 첫 번째 컬럼만 꺼내준다. 
    # 단일 컬럼 SELECT에서 유용하며, ORM에서 객체 자체를 받을 때 매우 자주 사용된다.
    names = conn.execute(select(users.c.name)).scalars().all()
    print('🟩', names)
    
    # 순회
    for row in conn.execute(select(users)):
        print('🟡', row.name)
        

2026-06-01 11:00:56,403 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 11:00:56,404 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.id = ?
2026-06-01 11:00:56,404 INFO sqlalchemy.engine.Engine [generated in 0.00137s] (1,)
🟩 (1, 'Alice', 'alice@example.com', datetime.datetime(2026, 6, 1, 1, 19, 48))
2026-06-01 11:00:56,406 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.id = ?
2026-06-01 11:00:56,406 INFO sqlalchemy.engine.Engine [cached since 0.003353s ago] (1,)
🟩 (1, 'Alice', 'alice@example.com', datetime.datetime(2026, 6, 1, 1, 19, 48))
2026-06-01 11:00:56,408 INFO sqlalchemy.engine.Engine SELECT users.name 
FROM users 
WHERE users.id = ?
2026-06-01 11:00:56,408 INFO sqlalchemy.engine.Engine [generated in 0.00064s] (1,)
🟩 Alice
2026-06-01 11:00:56,409 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FRO

### 9.7 종합 예제

In [29]:
from sqlalchemy import select

with engine.connect() as conn:
    # 이메일에 'example.com'이 포함된 사용자 중,
    # 최근 가입한 5명의 이름과 이메일
    stmt = (
        select(users)
        .where(users.c.email.like("%example.com"))
        .order_by(users.c.created_at.desc())
        .limit(5)
    )
    
    for row in conn.execute(stmt):
        print(f"🟩 {row.name} <{row.email}> {row.created_at}")

2026-06-01 11:06:24,379 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 11:06:24,380 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.email LIKE ? ORDER BY users.created_at DESC
 LIMIT ? OFFSET ?
2026-06-01 11:06:24,380 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ('%example.com', 5, 0)
🟩 Frank <frank@example.com> 2026-06-01 01:27:26
🟩 Charlie <charlie@example.com> 2026-06-01 01:23:48
🟩 Diana <diana@example.com> 2026-06-01 01:23:48
🟩 Eve <eve@example.com> 2026-06-01 01:23:48
🟩 Bob <bob@example.com> 2026-06-01 01:22:02
2026-06-01 11:06:24,383 INFO sqlalchemy.engine.Engine ROLLBACK


---

## 10장. SELECT 심화

#### assresses 테이블 생성

In [30]:
addresses = Table(
    "addresses",
    metadata,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("users.id"), nullable=False),
    Column("city", String(50)),
    Column("street", String(120)),
)
metadata.create_all(engine)

2026-06-01 11:07:48,551 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 11:07:48,552 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-01 11:07:48,552 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 11:07:48,553 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("addresses")
2026-06-01 11:07:48,553 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 11:07:48,554 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("addresses")
2026-06-01 11:07:48,555 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-01 11:07:48,556 INFO sqlalchemy.engine.Engine 
CREATE TABLE addresses (
	id INTEGER NOT NULL, 
	user_id INTEGER NOT NULL, 
	city VARCHAR(50), 
	street VARCHAR(120), 
	PRIMARY KEY (id), 
	FOREIGN KEY(user_id) REFERENCES users (id)
)


2026-06-01 11:07:48,556 INFO sqlalchemy.engine.Engine [no key 0.00040s] ()
2026-06-01 11:07:48,557 INFO sqlalchemy.engine.Engine COMMIT


### 10.1 JOIN

여러 테이블을 연결하려면 `join()` 또는 `select_from()`을 사용한다. 먼저 두 테이블을 정의한다.

`users`와 `addresses` 사이에 외래 키가 정의되어 있으면, SQLAlchemy는 자동으로 JOIN 조건을 추론할 수 있다.

In [32]:
# INNER JOIN (자동 추론)
stmt = select(users.c.name, addresses.c.city).join(addresses)
print(stmt)

SELECT users.name, addresses.city 
FROM users JOIN addresses ON users.id = addresses.user_id


In [34]:
# 명시적으로 조인 조건 지정
stmt = (
    select(users.c.name, addresses.c.city)
        .join(addresses, users.c.id == addresses.c.user_id)
)
print(stmt)

SELECT users.name, addresses.city 
FROM users JOIN addresses ON users.id = addresses.user_id


In [35]:
# LEFT OUTER JOIN은 `outerjoin()`이다.
stmt = select(users.c.name, addresses.c.city).outerjoin(addresses)
print(stmt)

SELECT users.name, addresses.city 
FROM users LEFT OUTER JOIN addresses ON users.id = addresses.user_id


#### 여러 테이블을 조인하려면 체이닝한다.

여러 테이블을 조인하려면 체이닝한다.
```python
stmt = (
    select(users.c.name, addresses.c.city, orders.c.total)
    .join(addresses)
    .join(orders, orders.c.user_id == users.c.id)
)
```

### 10.2 `select_from()`: FROM 절 명시

여러 컬럼이 같은 이름을 가지거나, FROM의 시작점을 명확히 지정해야 할 때 `select_from()`을 쓴다.

In [39]:
from sqlalchemy import select

stmt = (
    select(users.c.name, addresses.c.city)
    .select_from(users.join(addresses))
    
    # users.join(addresses)  # users JOIN addresses ON users.id = addresses.user_id
    
    
)
print(stmt)

SELECT users.name, addresses.city 
FROM users JOIN addresses ON users.id = addresses.user_id


`users.join(addresses)`는 조인된 FROM 절 자체를 표현한다. `select()` 인자가 아니라 `select_from()`의 인자로 들어간다.

### 10.3 별칭(Alias)

같은 테이블을 여러 번 조인하거나, 컬럼 이름 충돌을 피할 때 별칭을 사용한다.

from sqlalchemy import alias

```python
# 직원과 매니저가 같은 employees 테이블에 있을 때
manager = employees.alias("manager")

stmt = (
    select(employees.c.name, manager.c.name.label("manager_name"))
    .join(manager, employees.c.manager_id == manager.c.id)
)
```

`.label()`은 컬럼에 별칭을 주어 SELECT 결과에서 구분되게 한다.

### 10.4 집계 함수와 GROUP BY

집계 함수는 `func` 객체로 접근한다.

In [40]:
from sqlalchemy import func

In [42]:
stmt = select(func.count()).select_from(users)
print(stmt)


SELECT count(*) AS count_1 
FROM users


In [43]:
stmt = select(func.count(users.c.id))
print(stmt)


SELECT count(users.id) AS count_1 
FROM users


In [ ]:
stmt = 🔹TODO
print(stmt)


GROUP BY는 `group_by()`로 추가한다.

In [48]:
stmt = (
    select(
        addresses.c.city.label('도시'),
        func.count(addresses.c.id).label('user_count')
    )
    .group_by(addresses.c.city)
)
print(stmt)


SELECT addresses.city AS "도시", count(addresses.id) AS user_count 
FROM addresses GROUP BY addresses.city


HAVING 절은 `having()`이다.

In [49]:
stmt = (
    select(
        addresses.c.city,
        func.count(addresses.c.id).label("user_count"),
    )
    .group_by(addresses.c.city)
    .having(func.count(addresses.c.id) > 10)
)
print(stmt)


SELECT addresses.city, count(addresses.id) AS user_count 
FROM addresses GROUP BY addresses.city 
HAVING count(addresses.id) > :count_1


### 10.5 주요 SQL 함수 `func`

`func` 객체로 거의 모든 SQL 함수에 접근할 수 있다.

```python
func.count()         # COUNT
func.sum(column)     # SUM
func.avg(column)     # AVG
func.min(column)     # MIN
func.max(column)     # MAX
func.now()           # 현재 시각
func.coalesce(a, b)  # 첫 번째 NULL이 아닌 값
func.lower(column)   # 소문자 변환
func.length(column)  # 문자열 길이
func.date(column)    # 날짜 부분만 추출
```

SQLAlchemy는 함수 이름을 검사하지 않으므로, 데이터베이스에 존재하는 함수라면 무엇이든 호출 가능하다. PostgreSQL의 `jsonb_extract_path_text`도 `func.jsonb_extract_path_text(...)`로 부르면 된다.

### 10.6 서브쿼리

서브쿼리는 `subquery()`로 만든다.

`subquery()` 는 `SELECT` 문을 이름 붙은 파생 테이블(derived table)로 바꿔서, 
바깥 쿼리에서 일반 테이블처럼 참조할 수 있게 만들어 줍니다

=> **Select 를 FROM 절에 들어갈 수 있는 객체로 변환**

In [51]:
# 사용자별 주소 개수를 구한 서브쿼리
addr_count = (
    select(
        addresses.c.user_id,
        func.count().label("addr_count"),
    )
    .group_by(addresses.c.user_id)
    .subquery()
)
print(addr_count)

SELECT addresses.user_id, count(*) AS addr_count 
FROM addresses GROUP BY addresses.user_id


In [52]:
# 메인 쿼리에서 활용 서브쿼리도 `.c`로 컬럼에 접근한다. 
stmt = (
    select(users.c.name, addr_count.c.addr_count)
    .join(addr_count, users.c.id == addr_count.c.user_id)
)
print(stmt)


SELECT users.name, anon_1.addr_count 
FROM users JOIN (SELECT addresses.user_id AS user_id, count(*) AS addr_count 
FROM addresses GROUP BY addresses.user_id) AS anon_1 ON users.id = anon_1.user_id


In [53]:
# SELECT 절에서 단일 스칼라 값으로 서브쿼리를 쓰려면 `scalar_subquery()`를 사용한다.

latest_addr = (
    select(addresses.c.city)
    .where(addresses.c.user_id == users.c.id)
    .order_by(addresses.c.id.desc())
    .limit(1)
    .scalar_subquery()
)
print('🟦', latest_addr)

🟦 (SELECT addresses.city 
FROM addresses, users 
WHERE addresses.user_id = users.id ORDER BY addresses.id DESC
 LIMIT :param_1)


In [54]:
stmt = select(users.c.name, latest_addr.label("latest_city"))
print('🟦', stmt)


🟦 SELECT users.name, (SELECT addresses.city 
FROM addresses 
WHERE addresses.user_id = users.id ORDER BY addresses.id DESC
 LIMIT :param_1) AS latest_city 
FROM users


### 10.7 CASE 표현식

조건부 표현식은 `case()`로 만든다.

In [ ]:
from sqlalchemy import case

stmt = select(
    users.c.name,
    🔹TODO
)
print(stmt)


### 10.8 EXISTS

EXISTS 서브쿼리는 `.exists()`로 만든다.

In [55]:
addr_exists = (
    select(addresses.c.id)
    .where(addresses.c.user_id == users.c.id)
    .exists()
)
print(addr_exists)


EXISTS (SELECT addresses.id 
FROM addresses, users 
WHERE addresses.user_id = users.id)


In [56]:
stmt = select(users.c.name).where(addr_exists)
# 주소를 가진 사용자만 조회
print(stmt)

SELECT users.name 
FROM users 
WHERE EXISTS (SELECT addresses.id 
FROM addresses 
WHERE addresses.user_id = users.id)


---

## 11장. UPDATE와 DELETE

### 11.1 `update()` 기본

`update()` 함수로 UPDATE 문을 만든다. `.where()`로 조건을, `.values()`로 변경할 값을 지정한다.

📌**중요**: `.where()` 없는 UPDATE는 **모든 행을 갱신**한다. 매우 위험하므로 항상 WHERE 절을 빠뜨리지 않았는지 확인하자.

In [57]:
from sqlalchemy import update

with engine.connect() as conn:
    result = conn.execute(select(users).where(users.c.name == "Alice"))
    print('🟩update전', result.one_or_none())
    
    stmt = (
        update(users)
        .where(users.c.name == "Alice")
        .values(email = "alice.new@mail.com")
    )
    print('🟡', stmt)

    result = conn.execute(stmt)
    print('🟦', result.rowcount)  # 영향받은 행 수

    result = conn.execute(select(users).where(users.c.name == "Alice"))
    print('🟩update후', result.one_or_none())

2026-06-01 11:43:21,085 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 11:43:21,085 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.name = ?
2026-06-01 11:43:21,086 INFO sqlalchemy.engine.Engine [generated in 0.00112s] ('Alice',)
🟩update전 (1, 'Alice', 'alice@example.com', datetime.datetime(2026, 6, 1, 1, 19, 48))
🟡 UPDATE users SET email=:email WHERE users.name = :name_1
2026-06-01 11:43:21,089 INFO sqlalchemy.engine.Engine UPDATE users SET email=? WHERE users.name = ?
2026-06-01 11:43:21,089 INFO sqlalchemy.engine.Engine [generated in 0.00064s] ('alice.new@mail.com', 'Alice')
🟦 1
2026-06-01 11:43:21,090 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.created_at 
FROM users 
WHERE users.name = ?
2026-06-01 11:43:21,091 INFO sqlalchemy.engine.Engine [cached since 0.006088s ago] ('Alice',)
🟩update후 (1, 'Alice', 'alice.new@mail.com', datetime.datetime(2026, 6, 1, 1, 19, 48))
2026-06-

### 11.2 표현식을 값으로 사용하기

`.values()`에 컬럼 표현식을 넘기면 SQL 표현식 그대로 사용된다.

In [58]:
stmt = update(users).values(id = users.c.id + 1)
print(stmt)

UPDATE users SET id=(users.id + :id_1)


In [59]:
# 다른 컬럼의 값으로 갱신
stmt = update(users).values(email=users.c.name)
print(stmt)

UPDATE users SET email=users.name


In [60]:
# 함수 사용
stmt = update(users).values(created_at = func.now())
print(stmt)

UPDATE users SET created_at=now()


### 11.3 다중 행을 각기 다른 값으로 UPDATE

INSERT처럼 파라미터 리스트를 전달하면 **executemany** 스타일로 동작한다.

```python
with engine.begin() as conn:
    conn.execute(
        update(users).where(users.c.name == bindparam("old_name")),
        [
            {"old_name": "Alice", "email": "alice@new.com"},
            {"old_name": "Bob", "email": "bob@new.com"},
        ],
    )
```

`bindparam`을 명시적으로 사용해야 파라미터가 같은 이름과 충돌하지 않는다. 이런 패턴은 자주 쓰이지는 않으며, ORM의 일괄 UPDATE를 사용하는 편이 더 일반적이다.

### 11.4 `delete()` 기본

`delete()` 함수는 DELETE 문을 만든다. 구조는 UPDATE와 매우 유사하다.

📌UPDATE와 마찬가지로, **WHERE 없는 DELETE는 전체 행을 삭제**한다. 매우 위험하다.

In [61]:
from sqlalchemy import delete

stmt = delete(users).where(users.c.name == "Alice")
print('🟡', stmt)

with engine.begin() as conn:
    result = conn.execute(stmt)
    print('🟦', result.rowcount)

🟡 DELETE FROM users WHERE users.name = :name_1
2026-06-01 12:03:18,205 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 12:03:18,206 INFO sqlalchemy.engine.Engine DELETE FROM users WHERE users.name = ?
2026-06-01 12:03:18,206 INFO sqlalchemy.engine.Engine [generated in 0.00040s] ('Alice',)
🟦 1
2026-06-01 12:03:18,207 INFO sqlalchemy.engine.Engine COMMIT


### 11.5 RETURNING과 함께 사용하기

UPDATE와 DELETE 모두 `.returning()`을 지원한다(지원되는 백엔드에서, 💢 MySQL 지원안함).

In [ ]:
# UPDATE ... RETURNING
stmt = (
    update(users)
    .where(users.c.name == "Alice")
    .values(email="alice.v2@example.com")
    🔹TODO
)
print('🟡', stmt)

with engine.begin() as conn:
    result = conn.execute(stmt)
    for row in result:  # MySQL 지원안함
        print('🟦', row.id, row.email)


```python
# DELETE ... RETURNING
stmt = (
    delete(users)
    .where(users.c.created_at < some_old_date)
    .returning(users.c.id, users.c.name)
)

with engine.begin() as conn:
    deleted = conn.execute(stmt).all()
    print(f"{len(deleted)}개의 사용자가 삭제됨")
```

이는 "갱신/삭제하면서 동시에 영향받은 행을 확인"하는 패턴에 매우 유용하다. 별도의 SELECT가 필요 없어진다.

### 11.6 영향받은 행 수: `rowcount`

UPDATE/DELETE 후 `result.rowcount`로 영향받은 행 수를 알 수 있다.

```python
with engine.begin() as conn:
    result = conn.execute(
        update(users).where(users.c.id == 999).values(name="Nobody")
    )
    if result.rowcount == 0:
        print("해당 사용자가 없습니다")
    else:
        print(f"{result.rowcount}개의 행이 갱신됨")
```

주의할 점이 있다. `rowcount`는 DBAPI 드라이버에 따라 동작이 조금씩 다르다. RETURNING이나 executemany를 사용할 때는 `-1`이 반환될 수도 있다. 정확한 행 수가 필요하다면 RETURNING을 사용해 직접 세는 편이 안전하다.

### 11.7 UPDATE FROM과 상관 서브쿼리

PostgreSQL, MySQL 등에서는 UPDATE 시 다른 테이블의 값을 참조할 수 있다.

```python
# 상관 서브쿼리로 UPDATE
latest_email = (
    select(addresses.c.city)
    .where(addresses.c.user_id == users.c.id)
    .order_by(addresses.c.id.desc())
    .limit(1)
    .scalar_subquery()
)

stmt = update(users).values(last_known_city=latest_email)
```

`UPDATE ... FROM` 문법은 SQLAlchemy가 백엔드에 따라 자동으로 생성한다. 별도 설정 없이 다중 테이블을 `where()` 안에서 참조하면 된다.

```python
stmt = (
    update(users)
    .where(addresses.c.user_id == users.c.id)
    .where(addresses.c.city == "Seoul")
    .values(is_seoul_user=True)
)
```

### 11.8 종합 예제: 비활성 사용자 정리

지금까지 배운 내용을 종합한 실전 시나리오다.

```python
from datetime import datetime, timedelta
from sqlalchemy import select, update, delete, insert, func

cutoff = datetime.utcnow() - timedelta(days=365)

with engine.begin() as conn:
    # 1) 비활성 사용자를 archived_users로 백업 (INSERT FROM SELECT)
    select_stmt = select(
        users.c.id, users.c.name, users.c.email
    ).where(users.c.last_login < cutoff)
    
    backup_stmt = insert(archived_users).from_select(
        ["id", "name", "email"], select_stmt
    )
    conn.execute(backup_stmt)
    
    # 2) 해당 사용자들을 비활성 상태로 표시 (UPDATE with RETURNING)
    update_stmt = (
        update(users)
        .where(users.c.last_login < cutoff)
        .values(is_active=False)
        .returning(users.c.id)
    )
    deactivated_ids = conn.execute(update_stmt).scalars().all()
    print(f"{len(deactivated_ids)}명의 사용자를 비활성화함")
    
    # 3) 2년 이상 로그인 없는 비활성 사용자는 완전 삭제
    deep_cutoff = datetime.utcnow() - timedelta(days=730)
    delete_stmt = (
        delete(users)
        .where(users.c.is_active == False)
        .where(users.c.last_login < deep_cutoff)
        .returning(users.c.id)
    )
    deleted_ids = conn.execute(delete_stmt).scalars().all()
    print(f"{len(deleted_ids)}명의 사용자를 영구 삭제함")
    
    # 트랜잭션이 블록 종료 시 자동 커밋됨
```

이 모든 작업이 **하나의 트랜잭션**에서 실행된다. 중간에 실패하면 전체가 롤백되어 데이터 일관성이 보장된다.

---

## Part 3 마무리

여기까지 따라왔다면 다음을 할 수 있게 되었다.

- `insert()`, `select()`, `update()`, `delete()`로 SQL을 Python 객체로 조립함
- 다중 행 INSERT와 RETURNING을 통해 효율적으로 데이터를 다룸
- WHERE, ORDER BY, LIMIT, JOIN, GROUP BY, HAVING 등 SQL의 주요 절을 표현함
- 서브쿼리, 별칭, CASE 표현식, EXISTS 등 복잡한 패턴을 다룸

이 모든 것이 ORM 없이 **Core만으로** 가능하다는 점이 중요하다. 많은 사람들이 SQLAlchemy를 "ORM 라이브러리"로만 알고 있지만, Core 자체로도 강력한 SQL 툴킷이다.

다음 Part 4부터는 본격적으로 **ORM**의 세계로 들어간다. Python 클래스로 데이터베이스 테이블을 표현하고, 객체 단위로 데이터를 다루는 방법을 배운다. Core에서 익힌 `select()`, `insert()`, `update()`, `delete()` 구문이 ORM에서도 그대로 사용되므로, 지금까지 배운 것이 헛되지 않다.
